# 59 Three-way face detectability comparison

Run MediaPipe Face Detector on three mesh variants per subject, all built from the same pipeline run so the only factor that varies across columns is the operator applied: the original head-isolated CTF surface, the vertex-deletion output of the shipped pipeline, and the rejected noise-perturbation strawman (Cotangent-Laplacian neighbour pull + per-vertex Gaussian offsets, with a separate boundary-transition smoothing pass).

**Cohort.** Iterates over the eleven valid thesis subjects: the optode-cap cohort S1--S7 (Subject 16--22) and the bare-cap cohort S8--S11 (Subject 12--15). Subject 11 is excluded (unusable raw scan). The output CSV carries the shared `s_id` and `optode` columns so the LaTeX table can render the cohort split and a per-cohort summary directly.

Populates §4.4.4 (Table 4.7) of the thesis as the three-way extension of notebook 56. The colour contact sheet is restricted to S2 (Subject 17) under the data-sharing rule.

Output: `thesis_results_out/detectability_comparison_summary.csv`.

In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    is_optode, s_id,
    SUBJECTS, load_raw, load_landmarks, run_pipeline,
    noise_perturb_with_taper,
)

import numpy as np
import pandas as pd
import pyvista as pv
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_py
from mediapipe.tasks.python import vision as mp_vision

import cedalion.dataclasses as cdc
from cedalion.vtktutils import trimesh_to_vtk_polydata

pv.OFF_SCREEN = True

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)
VIEW_ROOT = OUT_DIR / 'detectability_views_comparison'
VIEW_ROOT.mkdir(parents=True, exist_ok=True)

THESIS_FIG_DIR = pathlib.Path(
    '/home/ma7/BA/Thesis_template-master-LaTeX_template_thesis/'
    'LaTeX_template_thesis/Figures/results'
)

YAWS    = list(range(-90, 91, 30))    # 7 yaws
PITCHES = [-20, 0, 20]                # 3 pitches
WINDOW  = (640, 640)
GREY    = (200, 200, 200)
CAM_DISTANCE_MM = 700.0
ZOOM = 1.3

MODEL_PATH = str((pathlib.Path('models') / 'blaze_face_short_range.tflite').resolve())
_fd = mp_vision.FaceDetector.create_from_options(
    mp_vision.FaceDetectorOptions(
        base_options=mp_py.BaseOptions(model_asset_path=MODEL_PATH),
        min_detection_confidence=0.5,
    )
)


def _vertex_rgba(mesh):
    """Return (n,4) uint8 vertex colors if the mesh carries them, else None."""
    visual = mesh.visual
    if hasattr(visual, 'to_color'):
        try:
            v = visual.to_color()
            vc = getattr(v, 'vertex_colors', None)
            if vc is not None and len(vc):
                return np.asarray(vc, dtype=np.uint8)
        except Exception:
            pass
    vc = getattr(visual, 'vertex_colors', None)
    if vc is not None and len(vc):
        return np.asarray(vc, dtype=np.uint8)
    return None


def _wrap_with_color(mesh):
    """Wrap a trimesh into a pyvista PolyData, attaching RGB if available."""
    poly = pv.wrap(trimesh_to_vtk_polydata(mesh))
    rgba = _vertex_rgba(mesh)
    if rgba is not None and len(rgba) == poly.n_points:
        poly['RGB'] = rgba[:, :3]
    return poly


def render_views(surface, out_dir, tag, use_color):
    """Render the mesh from the yaw/pitch sweep at fixed 700 mm distance.

    yaw=0/pitch=0 is a true frontal view in the CTF frame: camera placed on
    the +X axis (anterior), looking toward the head centroid, +Z up.

    use_color=True paints the mesh with its per-vertex RGB if available
    (used for the contact-sheet renders so the figure matches the textured
    hero PNGs); use_color=False paints the mesh flat grey (used for the
    detector sweep so noise/delete/original are compared on the same
    template-matching baseline).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    poly = _wrap_with_color(surface.mesh) if use_color else pv.wrap(
        trimesh_to_vtk_polydata(surface.mesh)
    )
    has_color = use_color and 'RGB' in poly.array_names
    focal = np.asarray(surface.mesh.vertices).mean(axis=0)
    files = []
    for yaw in YAWS:
        for pitch in PITCHES:
            yaw_r, pitch_r = np.deg2rad(yaw), np.deg2rad(pitch)
            cam_dir = np.array([
                np.cos(yaw_r) * np.cos(pitch_r),
                np.sin(yaw_r) * np.cos(pitch_r),
                np.sin(pitch_r),
            ])
            p = pv.Plotter(off_screen=True, window_size=WINDOW)
            if has_color:
                p.add_mesh(poly, scalars='RGB', rgb=True, smooth_shading=True)
            else:
                p.add_mesh(poly, color=[c/255 for c in GREY], smooth_shading=True)
            p.set_background('white')
            p.enable_anti_aliasing('ssaa')
            p.camera.position    = tuple(focal + CAM_DISTANCE_MM * cam_dir)
            p.camera.focal_point = tuple(focal)
            p.camera.up          = (0.0, 0.0, 1.0)
            p.camera.zoom(ZOOM)
            fn = out_dir / f'{tag}_yaw{yaw:+04d}_pitch{pitch:+03d}.png'
            p.screenshot(str(fn))
            p.close()
            files.append((yaw, pitch, fn))
    return files


def detect(image_path):
    img = cv2.imread(str(image_path))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = _fd.detect(mp_img)
    if not res.detections:
        return 0, 0.0
    confs = [d.categories[0].score for d in res.detections]
    return len(confs), float(max(confs))

I0000 00:00:1777403114.364928   24802 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1777403114.387715   24827 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.4-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 6.19.11-arch1-1)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777403114.395975   24804 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
CONTACT_SUBJECT = 12  # only S2's color renders are produced (data-sharing rule)
COLOR_VIEW_ROOT = OUT_DIR / 'detectability_views_comparison_color'
COLOR_VIEW_ROOT.mkdir(parents=True, exist_ok=True)

rows = []
for n in SUBJECTS:
    print(f'\n=== Subject{n} ===', flush=True)
    raw = load_raw(n)
    lm  = load_landmarks(n)
    art = run_pipeline(raw, lm, subject=n)

    lm_ctf_xyz = art.landmarks_ctf.pint.dequantify().values

    # noise variant: cotangent-Laplacian relaxation + per-vertex Gaussian
    # offsets, then a separate boundary-transition smoothing pass. Same code
    # path as the original anonymizer.NOISE method that produced the hero PNGs.
    noise_mesh = noise_perturb_with_taper(art.surface_ctf.mesh, art.mask, lm_ctf_xyz)
    surface_noise = cdc.TrimeshSurface(
        mesh=noise_mesh,
        crs=art.surface_ctf.crs,
        units=art.surface_ctf.units,
    )

    variants = {
        'original': art.surface_ctf,
        'delete':   art.surface_anon_ctf,
        'noise':    surface_noise,
    }

    for tag, surface in variants.items():
        # Detector sweep: grey, neutral lighting, identical across variants.
        sub_dir = VIEW_ROOT / f'subject{n}' / tag
        files = render_views(surface, sub_dir, tag, use_color=False)
        hits = 0
        max_conf = 0.0
        for yaw, pitch, fn in files:
            k, c = detect(fn)
            if k:
                hits += 1
            if c > max_conf:
                max_conf = c
        rows.append({
            's_id':     s_id(n),
            'subject':  n,
            'optode':   is_optode(n),
            'method':   tag,
            'n_views':  len(files),
            'hits':     hits,
            'max_conf': max_conf,
        })
        print(f'  {tag:10s}  hits={hits:2d}/{len(files):2d}  max_conf={max_conf:.3f}', flush=True)

        # Contact-sheet renders: color, only for the representative subject.
        if n == CONTACT_SUBJECT:
            color_dir = COLOR_VIEW_ROOT / f'subject{n}' / tag
            render_views(surface, color_dir, tag, use_color=True)


=== Subject16 ===
  original    hits= 6/21  max_conf=0.808
  delete      hits= 2/21  max_conf=0.648
  noise       hits= 2/21  max_conf=0.753

=== Subject17 ===
  original    hits= 3/21  max_conf=0.820
  delete      hits= 1/21  max_conf=0.519
  noise       hits= 0/21  max_conf=0.000

=== Subject18 ===
  original    hits= 3/21  max_conf=0.722
  delete      hits= 0/21  max_conf=0.000
  noise       hits= 0/21  max_conf=0.000

=== Subject19 ===
  original    hits= 4/21  max_conf=0.774
  delete      hits= 0/21  max_conf=0.000
  noise       hits= 0/21  max_conf=0.000

=== Subject20 ===
  original    hits= 2/21  max_conf=0.670
  delete      hits= 0/21  max_conf=0.000
  noise       hits= 0/21  max_conf=0.000

=== Subject21 ===
  original    hits= 7/21  max_conf=0.873
  delete      hits= 1/21  max_conf=0.522
  noise       hits= 0/21  max_conf=0.000

=== Subject22 ===
  original    hits= 7/21  max_conf=0.863
  delete      hits= 0/21  max_conf=0.000
  noise       hits= 0/21  max_conf=0.000

=== S

In [3]:
df = pd.DataFrame(rows)
if len(df) and 's_id' in df.columns:
    df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
csv = OUT_DIR / 'detectability_comparison_summary.csv'
df.to_csv(csv, index=False)
print(f'Wrote {csv}\n')

# Format like Table 4.4 of the thesis for visual comparison.
wide = df.pivot(index='subject', columns='method', values=['hits', 'max_conf'])
wide.columns = [f'{m}_{k}' for k, m in wide.columns]
wide = wide.reset_index().sort_values('subject').reset_index(drop=True)

# Re-label subjects to S1..S7 in the same fixed order used in the thesis.
S_MAP = {n: f'S{i+1}' for i, n in enumerate(sorted(df['subject'].unique()))}
wide.insert(0, 'ID', wide['subject'].map(S_MAP))
wide = wide.drop(columns=['subject'])
print(wide.to_string(index=False))

Wrote thesis_results_out/detectability_comparison_summary.csv

 ID  delete_hits  noise_hits  original_hits  delete_max_conf  noise_max_conf  original_max_conf
 S1         15.0        13.0           18.0         0.819094        0.797325           0.945619
 S2          3.0         4.0           10.0         0.548038        0.829374           0.876601
 S3         11.0         7.0           17.0         0.718792        0.679603           0.890983
 S4          1.0         2.0           16.0         0.551863        0.678810           0.963591
 S5          2.0         2.0            6.0         0.648316        0.752748           0.807684
 S6          1.0         0.0            3.0         0.518890        0.000000           0.819893
 S7          0.0         0.0            3.0         0.000000        0.000000           0.721627
 S8          0.0         0.0            4.0         0.000000        0.000000           0.773727
 S9          0.0         0.0            2.0         0.000000        0.000

In [4]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Thesis data-sharing rule: only Subject 17 (S2) appears in rendered figures.
TAGS  = ['original', 'delete', 'noise']
ROW_TITLES = ['Original', 'Vertex deletion', 'Noise perturbation']

fig, axes = plt.subplots(
    len(TAGS), len(YAWS), figsize=(2.4 * len(YAWS), 2.4 * len(TAGS))
)
for r, tag in enumerate(TAGS):
    for c, yaw in enumerate(YAWS):
        ax = axes[r, c]
        img = mpimg.imread(
            COLOR_VIEW_ROOT / f'subject{CONTACT_SUBJECT}' / tag /
            f'{tag}_yaw{yaw:+04d}_pitch+00.png'
        )
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(f'yaw {yaw:+d}°', fontsize=9)
        if c == 0:
            ax.set_ylabel(ROW_TITLES[r], fontsize=10)
fig.tight_layout()

out = THESIS_FIG_DIR / 'detectability_contact_S2.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')

Wrote /home/ma7/BA/Thesis_template-master-LaTeX_template_thesis/LaTeX_template_thesis/Figures/results/detectability_contact_S2.png
